# 第五章：细胞类型注释（手动 + CellTypist）

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

前四篇我们完成了"技术层面"的数据处理：QC、归一化、降维、聚类、批次矫正。得到了 20 个 Leiden cluster——但这些 cluster 只是无名的数字编号（0, 1, 2, ...）。

从这一篇开始，我们进入"生物学层面"。细胞注释（cell type annotation）是 scRNA-seq 分析中最核心、也是最考验功底的一步。它回答的是：每个 cluster 是什么细胞类型？

这个问题看似简单，实则充满陷阱。同一个 cluster 可能是一种纯粹的细胞类型，也可能是两种相似细胞的混合。同一种细胞类型可能被分成了多个 cluster（过度聚类），也可能和其他类型混在了一起（聚类不足）。

我们的策略是三层注释法：先用 canonical markers 做手动粗注释，再用 CellTypist 做自动化验证，最后对关键大类做亚群细分。

## 注释策略概述

### 为什么不能只靠自动注释

CellTypist、SingleR、Azimuth 等自动注释工具很方便，但它们有几个根本限制：

模型偏差：CellTypist 的 Immune_All_Low 模型是在免疫细胞数据上训练的——对肝脏中的非免疫细胞（肝细胞、内皮细胞、星状细胞）不了解。
组织特异性：不同组织中"同一种细胞"的表达特征可能不同。比如肝脏的 Kupffer 细胞和血液的巨噬细胞虽然都是巨噬细胞，但 marker 表达有显著差异。
黑盒问题：自动注释给出一个标签，但不告诉你"为什么"。如果注释错了，你很难发现。

### 为什么不能只靠手动注释

纯手动注释同样有问题：

主观偏差：不同分析者可能对同一个 cluster 给出不同的标签
遗漏盲区：你可能不熟悉某些稀有细胞类型的 marker
耗时：对 20 个 cluster 逐一检查 marker，需要大量时间

最佳实践是两者结合：手动注释为主（因为你了解组织生物学），自动注释为辅（帮你发现盲区）。

## Part A：最优 Resolution 选择

### 选多少个 cluster 来注释

在开始注释之前，先确定用哪个 resolution 的聚类结果。前面我们跑了 res=0.2 到 res=2.0 的一系列聚类。怎么选？

我们用两个定量指标辅助决策：

In [ ]:
import scanpy as sc
import numpy as np
from sklearn.metrics import adjusted_rand_score, silhouette_score

adata = sc.read_h5ad("results/03_integrated.h5ad")

resolutions = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
res_keys = [f"leiden_harmony_{r}" for r in resolutions]

# ARI：相邻 resolution 的聚类一致性
for i in range(len(resolutions) - 1):
    ari = adjusted_rand_score(
        adata.obs[res_keys[i]].values,
        adata.obs[res_keys[i+1]].values
    )
    print(f"  ARI({resolutions[i]} → {resolutions[i+1]}): {ari:.4f}")

# Silhouette score（采样加速）
np.random.seed(42)
sample_idx = np.random.choice(adata.n_obs, 5000, replace=False)
X_sample = adata.obsm["X_pca_harmony"][sample_idx]

for res, key in zip(resolutions, res_keys):
    labels = adata.obs[key].values[sample_idx].astype(int)
    sil = silhouette_score(X_sample, labels)
    print(f"  res={res}: silhouette={sil:.4f}")

图 1：不同分辨率下的聚类稳定性分析

ARI（Adjusted Rand Index） 衡量两个聚类之间的一致性。当 ARI 从一个高值急剧下降时，说明进一步增加 resolution 导致了"不稳定"的分裂——新产生的 cluster 可能不可靠。

Silhouette Score 衡量每个细胞和它所属 cluster 的"紧凑度"vs 和其他 cluster 的"分离度"。越高说明聚类越好。

综合来看，我们选择 res=0.8（20 个 cluster） 作为注释起点。

## Part B：Marker 基因检测 + 手动注释

### 计算每个 cluster 的 marker genes

In [ ]:
# 检测每个 cluster 的差异表达基因
sc.tl.rank_genes_groups(
    adata, groupby="leiden_anno",
    method="wilcoxon", use_raw=True, pts=True
)

# 打印 top 5 markers
markers_df = sc.get.rank_genes_groups_df(adata, group=None)
for cl in sorted(adata.obs["leiden_anno"].unique(), key=int):
    sub = markers_df[markers_df["group"] == cl].head(5)
    genes = ", ".join(sub["names"].values)
    print(f"  Cluster {cl}: {genes}")

这些 marker genes 是我们判断细胞类型的第一手证据。比如如果一个 cluster 的 top markers 是 CD3D, CD3E, TRAC——不用想了，这是 T 细胞。

### Canonical marker 基因

肝脏中主要细胞类型的 canonical marker：

In [ ]:
marker_dict = {
    # 免疫细胞 (CD45+/PTPRC+)
    "T cells":      ["CD3D", "CD3E", "TRAC"],
    "NK cells":     ["NKG7", "KLRD1", "GNLY", "KLRF1"],
    "B cells":      ["CD79A", "MS4A1", "CD19"],
    "Plasma":       ["JCHAIN", "MZB1", "SDC1"],
    "Monocytes":    ["CD14", "S100A8", "S100A9"],
    "Macrophages":  ["CD68", "MARCO", "C1QA", "C1QB"],
    "cDC":          ["FCER1A", "CD1C", "CLEC10A"],
    "pDC":          ["LILRA4", "IRF7", "CLEC4C"],
    "Mast cells":   ["TPSAB1", "CPA3", "KIT"],
    # 非免疫细胞 (CD45-)
    "Hepatocytes":  ["ALB", "APOB", "CYP3A4", "SERPINA1"],
    "Cholangiocytes": ["EPCAM", "KRT19", "SOX9"],
    "Endothelial":  ["PECAM1", "CDH5", "VWF"],
    "HSCs":         ["ACTA2", "COL1A1", "DCN"],
    "Proliferating": ["MKI67", "TOP2A", "STMN1"],
}

图 2：大类细胞标记基因热图——每列一个 cluster，每行一个标记基因

### Dotplot：最直观的判断工具

In [ ]:
major_markers = [
    "CD3D", "CD4", "IL7R", "CD8A", "GZMK",  # T cells
    "NKG7", "KLRD1", "GNLY",                  # NK
    "CD79A", "MS4A1", "JCHAIN", "MZB1",       # B / Plasma
    "CD14", "S100A8", "FCGR3A",               # Monocytes
    "CD68", "C1QA",                            # Macrophages
    "FCER1A", "LILRA4",                        # DC
    "TPSAB1",                                  # Mast
    "ALB", "APOB",                             # Hepatocytes
    "EPCAM",                                   # Cholangiocytes
    "PECAM1", "CDH5", "CLEC4G",               # Endothelial
    "ACTA2", "COL1A1",                         # HSCs
    "MKI67", "TOP2A",                          # Proliferating
]
major_markers = [g for g in major_markers
                 if g in adata.raw.var_names]

sc.pl.dotplot(adata, var_names=major_markers,
              groupby="leiden_anno",
              standard_scale="var", use_raw=True, show=False)
plt.savefig("results/figures/04_dotplot_major_markers.png",
            dpi=150, bbox_inches="tight")
plt.close()

图 3：大类细胞标记基因气泡图

图 1：Canonical marker dotplot。 这是细胞注释中最有价值的图。点的大小表示该基因在多少比例的细胞中检测到（detection rate），颜色表示平均表达量（standard scaled）。观察要点：

CD3D 高表达的 cluster → T 细胞
NKG7 高但 CD3D 低的 cluster → NK 细胞
S100A8 + CD14 高 → 单核细胞
C1QA + CD68 高 → 巨噬细胞
ALB 高 → 肝细胞
PECAM1 + CDH5 高 → 内皮细胞

### 自动打分 + 手动修正

我们对每个 cluster 计算各 canonical marker set 的平均表达分，自动选出最匹配的细胞类型。然后人工审查歧义情况：

In [ ]:
# 大类注释映射
adata.obs["cell_type"] = adata.obs["leiden_anno"].map(
    broad_annotations
)
print(adata.obs["cell_type"].value_counts().to_string())

**输出：**

> 📊 输出：
> T cells            20409
> NK cells           10806
> Endothelial         7885
> Monocytes           4624
> Macrophages         3649
> Hepatocytes         3030
> HSCs/Mesenchyme     2260
> cDC                 2132
> B cells             1922
> Proliferating        789
> Plasma cells         661
> pDC                  568

12 种大类细胞类型，和肝脏的生物学预期一致。T 细胞是最大群体（20,409 个），这和肝脏作为免疫器官的特性一致——肝脏含有大量肝内 T 细胞。

图 4：大类注释 UMAP——12 种主要细胞类型

## Part C：CellTypist 自动注释

CellTypist（Domínguez Conde et al., Science, 2022）是一个基于 logistic regression 的自动注释工具。我们用两个分辨率的模型做交叉验证：

In [ ]:
import celltypist

# 准备输入（CellTypist 需要 log-normalized 数据）
adata_ct = adata.raw.to_adata()
adata_ct.obs = adata.obs.copy()

# 粗粒度模型
model_low = celltypist.models.Model.load(
    model="Immune_All_Low.pkl"
)
pred_low = celltypist.annotate(
    adata_ct, model=model_low, majority_voting=True
)
adata.obs["celltypist_low"] = \
    pred_low.predicted_labels["majority_voting"]

# 细粒度模型
model_high = celltypist.models.Model.load(
    model="Immune_All_High.pkl"
)
pred_high = celltypist.annotate(
    adata_ct, model=model_high, majority_voting=True
)
adata.obs["celltypist_high"] = \
    pred_high.predicted_labels["majority_voting"]

**输出：**

> 📊 输出：
> CellTypist Low 类别: 14
> CellTypist High 类别: 31

CellTypist 的 majority_voting=True 参数让它先对每个细胞独立预测，然后在 Leiden cluster 内做多数投票——这样每个 cluster 的注释更一致。

### 手动 vs 自动的交叉验证

In [ ]:
import pandas as pd

# 交叉表
ct_cross = pd.crosstab(
    adata.obs["cell_type"],
    adata.obs["celltypist_low"],
    normalize="index"
)
print(ct_cross.round(2).to_string())

看交叉表是验证注释一致性的最佳方式。如果手动注释的 "T cells" 在 CellTypist 中也大部分被标记为 "T cells"——好，两种方法一致。如果出现大面积"不一致"，就需要回去检查是哪个方法出了问题。

⚠️ 踩坑预警：CellTypist 对非免疫细胞的"误判"

Immune_All_Low 模型只包含免疫细胞类型。所以肝细胞、内皮细胞、星状细胞在 CellTypist 的输出中都会被"硬塞"到某个免疫细胞类别。比如内皮细胞可能被标为 "Endothelial cells"（如果模型恰好有这个类别）或者被随机分配到某个免疫类型。

教训：自动注释工具的输出不能盲信，必须和手动注释交叉验证。

图 6：各细胞类型的条件分布

## Part D：亚群细分注释

大类注释只是起点。对于临床研究来说，真正有价值的是亚群信息。比如，"巨噬细胞"这个大类里面可能包含了常驻 Kupffer 细胞、炎症巨噬细胞、TREM2+ SAMac——它们在疾病中的角色完全不同。

### 策略：提取 → 重聚类 → 亚群注释

对每个关键大类，我们提取该类型的所有细胞，重新做 HVG → PCA → Harmony → UMAP → Leiden，然后基于亚群 marker 进行细分注释。

为什么要重新做全流程？因为在全数据集的聚类中，同一大类内的亚群差异被压缩了——那些区分亚群的 HVG 在全局分析中可能排名很低。

In [ ]:
# 以 T 细胞为例
adata_t = adata[adata.obs["cell_type"] == "T cells"].copy()
adata_t_raw = adata_t.raw.to_adata()

# 重新选 HVG（子集专用）
sc.pp.highly_variable_genes(adata_t_raw, n_top_genes=2000,
                             flavor="seurat_v3",
                             batch_key="sample")
sc.pp.scale(adata_t_raw, max_value=10)
sc.tl.pca(adata_t_raw, n_comps=20)

# 子集 Harmony
import harmonypy as hm
ho_sub = hm.run_harmony(
    adata_t_raw.obsm["X_pca"][:, :20],
    adata_t_raw.obs, "sample",
    max_iter_harmony=20, random_state=42
)
adata_t_raw.obsm["X_pca_harmony"] = ho_sub.Z_corr

# 重新聚类
sc.pp.neighbors(adata_t_raw, use_rep="X_pca_harmony",
                n_neighbors=15, n_pcs=20)
sc.tl.umap(adata_t_raw, min_dist=0.3)
sc.tl.leiden(adata_t_raw, resolution=0.6,
            key_added="sub_leiden", flavor="igraph")

图 7：T 细胞亚群 subclustering

### T 细胞亚群注释

T 细胞的亚群 marker：

亚群
关键 Marker

CD4 naive
CD4, CCR7, LEF1, SELL, TCF7

CD4 memory
CD4, IL7R, S100A4

Treg
CD4, FOXP3, IL2RA, CTLA4

CD8 effector
CD8A, CD8B, GZMB, PRF1

CD8 memory
CD8A, CD8B, GZMK, EOMES

γδT
TRDC, TRGC1, TRDV2

MAIT
SLC4A10, TRAV1-2, KLRB1

NKT
CD3D, NKG7, KLRD1

图 8：T 细胞亚群标记基因

图 9：巨噬细胞亚群 subclustering

图 10：巨噬细胞亚群标记基因

图 11：内皮细胞亚群 subclustering

图 12：内皮细胞亚群标记基因

图 13：单核细胞亚群 subclustering

图 14：单核细胞亚群标记基因

图 15：NK 细胞亚群 subclustering

图 16：NK 细胞亚群标记基因

### 最终细注释结果

对 T 细胞、NK 细胞、单核/巨噬细胞、内皮细胞等关键大类都做了亚群细分后，我们得到 26 种细注释亚群：

**输出：**

> 📊 输出：
> CD56bright NK       7174
> CD4 naive           4815
> CD8 memory          4805
> Vascular EC         4193
> MAIT                4114
> CD56dim NK          3632
> CD4 memory          3542
> Hepatocytes         3030
> CD16 Mono           2603
> HSCs/Mesenchyme     2260
> cDC                 2132
> CD14 Mono           2021
> B cells             1922
> Kupffer cells       1899
> LSEC zone1          1836
> gdT                 1790
> Lymphatic EC        1789
> LAM                 1305
> CD8 effector        1014
> Proliferating        789
> Plasma cells         661
> pDC                  568
> Resident Mac         425
> Treg                 329
> LSEC zone2/3          67
> Inflammatory Mac      20

几个亮点值得关注：

MAIT 细胞（4,114 个） 是肝脏中最丰富的非常规 T 细胞之一，占全部 T 细胞的约 20%。这是肝脏特有的免疫特征。

LAM（Lipid-Associated Macrophages, 1,305 个） 对应原文中的 TREM2+CD9+ 瘢痕相关巨噬细胞（SAMac）——这是原文的核心发现之一。

Kupffer 细胞（1,899 个） vs LAM（1,305 个）：原文发现肝硬化中 Kupffer 细胞减少而 LAM 增加，我们的注释为后续的组成分析做好了准备。

LSEC zone2/3（67 个） 和 Inflammatory Mac（20 个） 数量很少——这可能是真实的稀有群体，也可能是注释噪声。后续分析中需要谨慎对待这些小群。

## 可视化

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(24, 9))
sc.pl.umap(adata, color="cell_type", ax=axes[0], show=False,
           title="Broad Cell Type (12 types)",
           frameon=False, legend_fontsize=8)
sc.pl.umap(adata, color="cell_type_fine", ax=axes[1], show=False,
           title="Fine Cell Type (26 subtypes)",
           frameon=False, legend_fontsize=6)
plt.tight_layout()
plt.savefig("results/figures/04_final_annotation_umap.png",
            dpi=150, bbox_inches="tight")
plt.close()

图 17：最终细胞注释 UMAP——26 种细胞亚型

图 2：最终注释 UMAP。 左图为 12 种大类注释，右图为 26 种细分注释。可以清楚地看到：T 细胞大群内部细分出了 CD4 naive、CD4 memory、CD8、MAIT、γδT 等多个亚群；内皮细胞分化出了 LSEC zone1、Vascular EC、Lymphatic EC 三种亚群。

## 与原文比较

📖 与 Ramachandran et al., 2019 对照

原文鉴定出的主要细胞类型包括：T/NK 细胞、巨噬细胞、内皮细胞、间充质细胞、B 细胞、肝细胞、胆管细胞等。我们的 12 种大类注释和原文高度一致。

关键对照点：

SAMac（LAM）：原文定义的 TREM2+CD9+ SAMac 对应我们的 "LAM" 亚群。我们的注释基于 TREM2/SPP1/GPNMB/CD9 的表达模式，这些 marker 和原文完全一致。
内皮亚群：原文报告了 LSEC 和 vascular endothelial cells 的区分，我们进一步分出了 zone1 LSEC、zone2/3 LSEC、vascular EC 和 lymphatic EC。
T 细胞亚群：原文没有对 T 细胞做详细亚群分析（不是他们的研究重点）。我们的 8 种 T 细胞亚群为后续的免疫分析提供了更精细的基础。

一个值得注意的差异：我们没有检测到胆管上皮细胞（cholangiocytes），而原文有。这可能是因为我们的聚类 resolution 下，少量胆管细胞被合并到了其他 cluster。在 res=1.2 以上可以看到 EPCAM+/KRT19+ 的小 cluster 出现，但细胞数太少（<50）不够稳健。

---

### 🔬 方法选择指南

🔬 方法选择指南：细胞注释工具对比
工具原理需要参考数据适用场景优缺点手动Marker ✓领域知识 + dotplot/violin可视化否（需先验知识）金标准，任何组织✅ 最可靠 ❌ 耗时、依赖专家CellTypist ✓逻辑回归分类器内置预训练模型免疫细胞（模型覆盖好）✅ 快速、有多层级模型 ❌ 非免疫细胞覆盖有限SingleRSpearman相关性匹配参考转录组需要参考数据集有高质量参考时✅ 无需训练 ❌ 依赖参考质量AzimuthSeurat参考映射(WNN)Seurat预建参考有Azimuth参考的组织(PBMC/肺等)✅ 交互式Web界面 ❌ 仅限R/特定组织scTypeMarker基因集加权评分需要marker基因集自定义marker评分✅ 简单透明 ❌ 需要高质量marker列表
我们的选择：手动 + CellTypist 三层验证。理由：

① 手动注释是不可替代的——无论自动工具多先进，最终的细胞身份判断都需要领域知识验证。我们通过canonical marker + dotplot建立"基线真相"；
② CellTypist作为验证而非替代——我们用CellTypist的两个模型（Low + High resolution）交叉验证手动结果，一致性>85%增加了信心；
③ 肝脏数据的特殊性——肝脏同时包含免疫细胞（CellTypist强项）和非免疫细胞（肝细胞、星状细胞等），纯靠自动注释会在非免疫细胞上犯错。

如何选择注释工具

• 你的组织有Azimuth参考吗？ → 如果有（如PBMC、肺、肾），优先使用Azimuth快速获得高置信度注释
• 你研究的主要是免疫细胞吗？ → CellTypist的Immune_All模型是目前最全面的免疫细胞分类器
• 你有高质量的参考数据集吗？ → SingleR不需要训练，直接匹配参考转录组
• 以上都没有？ → 回到手动注释——查文献、找canonical marker、用dotplot判断

核心原则：自动注释是起点而非终点。任何自动注释结果都必须通过marker基因表达模式来验证。如果自动工具说某个cluster是"CD8 T细胞"，但dotplot上CD8A/CD8B不表达，那就是错的。

---

## 方法论反思

### 注释的"不确定性"

细胞注释不是精确科学。几个需要坦诚面对的问题：

1. 边界是模糊的。 单核细胞和巨噬细胞之间没有绝对的分界线——它们是一个连续的分化谱系。我们的注释是在某个特定位置"画了一条线"。

2. 名字可以不同。 我们叫 "LAM"（Lipid-Associated Macrophages），原文叫 "SAMac"（Scar-Associated Macrophages）。它们指的是同一群细胞，只是从不同角度命名。在你的论文中，选择最符合你研究语境的名字即可。

3. 小群体的注释要谨慎。 Inflammatory Mac 只有 20 个细胞——这是真实的细胞亚群，还是聚类噪声？在报告中，对小群体的注释要加上"putative"（假定的）。

### 注释的最终检验

最好的验证方法不在计算层面，而在生物学层面：你的注释能不能解释已知的生物学现象？ 比如：

肝硬化中 Kupffer 细胞应该减少——我们的注释是否在组成分析中验证了这一点？
LAM/SAMac 应该表达 TREM2 和 SPP1——是这样吗？
肝细胞应该表达 ALB——它们确实在 ALB 高表达区域吗？

这些验证会在后续的差异分析和组成分析中逐一展开。

## 小结

这一篇我们完成了：

✅ 最优 resolution 选择（ARI + Silhouette，选定 res=0.8）
✅ Canonical marker 手动注释 → 12 种大类
✅ CellTypist 自动注释（Low + High 双分辨率）→ 交叉验证
✅ 亚群提取 + 重聚类 → 26 种细注释亚群
✅ 注释结果可视化：UMAP + dotplot + heatmap

关键数字：

大类注释：12 种（T cells, NK, Macrophages, Monocytes, Endothelial, Hepatocytes, HSCs/Mesenchyme, B cells, cDC, pDC, Plasma, Proliferating）
细分注释：26 种亚群
注释一致性：手动 vs CellTypist 的大类对应率 >85%

下一篇，我们将在注释好的数据上做差异表达分析——比较每种细胞类型在 Healthy vs Cirrhotic 条件下的基因表达变化。这是从"描述型分析"跨入"机制发现"的关键一步。